# Notebook 1: Database Setup & Pantry Initialization

## Purpose
This notebook sets up the SQLite pantry database, loads the 50-item JSON inventory,
and provides helper functions used by all other notebooks.

## Run this notebook first before any other notebook.

---

## 1.2 Imports

In [2]:
import mysql.connector
import json
import os
import sql_openai_config
from datetime import date, datetime
from pathlib import Path

# Paths — JSON file still used for seeding (optional now)
PROJECT_ROOT = Path(os.getcwd()).parent  # smart_pantry/
JSON_PATH = PROJECT_ROOT / 'data' / 'pantry_items.json'

MYSQL_CONFIG = sql_openai_config.get_mysql_config()


print(f"Project root : {PROJECT_ROOT}")
print(f"JSON path    : {JSON_PATH}")
print(f"MySQL DB     : {MYSQL_CONFIG['database']}")


Project root : /Users/nahlkhan/Desktop/smart_pantry 2
JSON path    : /Users/nahlkhan/Desktop/smart_pantry 2/data/pantry_items.json
MySQL DB     : smart_pantry


## 1.3 Database connection helper

In [3]:
def get_connection():
    """Return a MySQL connection to the pantry database."""
    return mysql.connector.connect(**MYSQL_CONFIG)

print("MySQL connection helper defined.")


MySQL connection helper defined.


## 1.4 Create the pantry table

The schema stores each pantry item with:
- `name` and `category` for identification
- `quantity` and `unit` for tracking how much is left
- `expiry_date` (ISO format) — the key field the agent reasons about
- `added_date` for auditing

In [4]:
def initialize_db():
    """Table already created in MySQL Workbench; nothing to do here."""
    print("Pantry table assumed to exist in MySQL (smart_pantry.pantry).")

initialize_db()


Pantry table assumed to exist in MySQL (smart_pantry.pantry).


## 1.5 Verify the loaded data

In [5]:
def show_pantry_table():
    """Display the full pantry table with days-until-expiry."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, category, quantity, unit, expiry_date FROM pantry ORDER BY expiry_date ASC"
    )
    rows = cursor.fetchall()
    conn.close()

    today = date.today()
    print(f"{'#':<4} {'Name':<20} {'Category':<12} {'Qty':>6} {'Unit':<8} {'Expiry':<12} {'Days left':>9}")
    print("-" * 78)
    for i, (name, cat, qty, unit, expiry) in enumerate(rows, 1):
        days = ""
        if expiry:
            d = expiry if isinstance(expiry, date) else datetime.strptime(str(expiry), "%Y-%m-%d").date()
            days = f"{(d - today).days:+d}"
        print(f"{i:<4} {name:<20} {cat:<12} {qty:>6} {unit:<8} {expiry:<12} {days:>9}")


In [6]:
show_pantry_table()


#    Name                 Category        Qty Unit     Expiry       Days left
------------------------------------------------------------------------------
1    eggs                 dairy           4.0 pieces   <12      -462
2    egg                  dairy           6.0 pieces   <12      -461
3    Ground turkey        Meat          500.0 grams    <12      -375
4    Avocado              Fruits          2.0 pieces   <12      -373
5    tomato               vegetable       4.0 pieces   <12       -11
6    banana               fruit           8.0 pieces   <12       -11
7    bread whole wheat    bakery          1.0 loaf     <12       -11
8    croissants           bakery          6.0 pieces   <12       -11
9    milk                 dairy         900.0 liter    <12       -10
10   salmon fillet        protein       400.0 grams    <12       -10
11   lettuce romaine      vegetable       2.0 heads    <12       -10
12   mushrooms button     vegetable     400.0 grams    <12       -10
13   raspberrie

## 1.6 Quick stats — items by category and expiry urgency

In [7]:
def pantry_stats():
    """Print a summary of pantry health."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, expiry_date FROM pantry"
    )
    rows = cursor.fetchall()
    conn.close()

    today  = date.today()
    expired = []
    urgent  = []  # 0–3 days
    soon    = []  # 4–7 days
    fine    = []  # 8+ days

    for name, expiry in rows:
        if expiry:
            d = expiry if isinstance(expiry, date) else datetime.strptime(str(expiry), "%Y-%m-%d").date()
            days = (d - today).days
            if days < 0:
                expired.append(name)
            elif days <= 3:
                urgent.append(name)
            elif days <= 7:
                soon.append(name)
            else:
                fine.append(name)

    print('=== Pantry Health Summary ===')
    print(f' Total items : {len(rows)}')
    print(f' EXPIRED     : {len(expired)} → {expired}')
    print(f' AT RISK (0–3d): {len(urgent)} → {urgent}')
    print(f' SOON (4–7d) : {len(soon)} → {soon}')
    print(f' OK (8d+)    : {len(fine)}')


In [8]:
pantry_stats()

=== Pantry Health Summary ===
 Total items : 182
 EXPIRED     : 44 → ['milk', 'tomato', 'yogurt', 'cream', 'chicken breast', 'chicken thighs', 'ground beef', 'pork chops', 'salmon fillet', 'tilapia fillets', 'tofu firm', 'cauliflower', 'kale', 'lettuce romaine', 'bell pepper red', 'bell pepper green', 'bell pepper yellow', 'cucumber', 'zucchini', 'eggplant', 'mushrooms button', 'mushrooms cremini', 'banana', 'grapes', 'strawberries', 'blueberries', 'raspberries', 'mango', 'kiwi', 'bread whole wheat', 'bread sourdough', 'burger buns', 'tortillas wheat', 'naan', 'pita bread', 'bagels', 'croissants', 'orange juice', 'apple juice', 'Avocado', 'Ground turkey', 'eggs', 'egg', 'eggs']
 AT RISK (0–3d): 9 → ['butter', 'paneer', 'tempeh', 'orange', 'pineapple', 'pear', 'tortillas corn', 'wraps spinach', 'wraps tomato basil']
 SOON (4–7d) : 1 → ['lime']
 OK (8d+)    : 128


## 1.7 Reset helper (re-run if you want a clean slate)

In [9]:
def reset_and_reload():
    """Drop all pantry rows and reload from JSON. Useful between experiments."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT ... FROM pantry ...")
    rows = cursor.fetchall()
    conn.close()

    load_from_json(JSON_PATH, clear_existing=False)
    print('Pantry reset and reloaded.')

# Uncomment to run:
# reset_and_reload()